<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 31 - Evaluación RA2: Ecosistemas Big Data

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Instrucciones

Este notebook es la **prueba práctica** del RA2. Completa todos los ejercicios.

- Lee cada enunciado con atención
- Escribe tu código en las celdas indicadas
- Ejecuta cada celda para verificar que funciona
- **Tiempo total:** 60 minutos

---

## Ejercicio 1: HDFS - Cálculo de bloques y replicación (15 puntos)

Dado un clúster HDFS con las siguientes características:
- **4 DataNodes** con 1 TB de almacenamiento cada uno
- **Tamaño de bloque:** 128 MB
- **Factor de replicación:** 3

Implementa una función que, dado el tamaño de un fichero en MB, calcule:
1. Número de bloques necesarios
2. Total de réplicas en el clúster
3. Espacio total ocupado (contando réplicas)
4. Si el fichero cabe en el clúster

In [ ]:
import math

def analizar_hdfs(tamano_fichero_mb, num_datanodes=4, tamano_bloque_mb=128,
                  factor_replicacion=3, almacenamiento_por_nodo_tb=1):
    """
    Analiza cómo se almacenaría un fichero en HDFS.
    """
    num_bloques = math.ceil(tamano_fichero_mb / tamano_bloque_mb)
    total_replicas = num_bloques * factor_replicacion
    espacio_total_mb = total_replicas * tamano_bloque_mb
    espacio_total_gb = espacio_total_mb / 1024
    capacidad_cluster_gb = num_datanodes * almacenamiento_por_nodo_tb * 1024
    cabe = espacio_total_gb <= capacidad_cluster_gb

    return {
        'bloques': num_bloques,
        'replicas_totales': total_replicas,
        'espacio_total_mb': espacio_total_mb,
        'espacio_total_gb': round(espacio_total_gb, 2),
        'capacidad_cluster_gb': capacidad_cluster_gb,
        'cabe_en_cluster': cabe
    }

# Pruebas
for tamano in [500, 5000, 50000, 500000, 2000000]:
    r = analizar_hdfs(tamano)
    print(f"\nFichero de {tamano:>10,} MB ({tamano/1024:.1f} GB):")
    print(f"  Bloques: {r['bloques']:,}")
    print(f"  Réplicas totales: {r['replicas_totales']:,}")
    print(f"  Espacio con réplicas: {r['espacio_total_gb']:,.2f} GB")
    print(f"  Capacidad clúster: {r['capacidad_cluster_gb']:,} GB")
    print(f"  ¿Cabe? {'✅ Sí' if r['cabe_en_cluster'] else '❌ No'}")

---

## Ejercicio 2: MapReduce - Análisis de logs (20 puntos)

Implementa un pipeline MapReduce completo para analizar logs de un servidor web.

Dado un conjunto de logs con formato `IP TIMESTAMP URL STATUS_CODE`, calcula:
1. **Conteo de visitas por URL** (Map + Reduce)
2. **Top 5 URLs más visitadas**
3. **Conteo de errores (status >= 400) por IP**

In [ ]:
import random
from collections import defaultdict
from datetime import datetime, timedelta

# Generar datos de ejemplo
random.seed(42)
ips = [f"192.168.1.{i}" for i in range(1, 21)]
urls = ["/index.html", "/products", "/api/users", "/api/orders",
        "/login", "/dashboard", "/api/search", "/about", "/contact", "/admin"]
status_codes = [200]*70 + [301]*5 + [404]*15 + [500]*10

base_time = datetime(2026, 1, 15, 8, 0, 0)
logs = []
for i in range(10000):
    ip = random.choice(ips)
    ts = base_time + timedelta(seconds=random.randint(0, 28800))
    url = random.choice(urls)
    status = random.choice(status_codes)
    logs.append(f"{ip} {ts.isoformat()} {url} {status}")

print(f"Total logs generados: {len(logs)}")
print(f"\nEjemplos:")
for log in logs[:5]:
    print(f"  {log}")

In [ ]:
# --- FASE MAP ---

def map_visitas_por_url(log_line):
    """Emite (url, 1) para cada línea de log."""
    parts = log_line.split()
    url = parts[2]
    return (url, 1)

def map_errores_por_ip(log_line):
    """Emite (ip, 1) solo si el status es >= 400."""
    parts = log_line.split()
    ip = parts[0]
    status = int(parts[3])
    if status >= 400:
        return (ip, 1)
    return None

# --- FASE SHUFFLE ---

def shuffle(mapped_pairs):
    """Agrupa valores por clave."""
    groups = defaultdict(list)
    for pair in mapped_pairs:
        if pair is not None:
            key, value = pair
            groups[key].append(value)
    return dict(groups)

# --- FASE REDUCE ---

def reduce_sum(key, values):
    """Suma todos los valores de una clave."""
    return (key, sum(values))

# === EJECUCIÓN COMPLETA ===

# 1. Visitas por URL
print("=== VISITAS POR URL ===")
mapped_urls = [map_visitas_por_url(log) for log in logs]
shuffled_urls = shuffle(mapped_urls)
reduced_urls = [reduce_sum(k, v) for k, v in shuffled_urls.items()]
reduced_urls.sort(key=lambda x: x[1], reverse=True)

for url, count in reduced_urls:
    print(f"  {url:20s} → {count:,} visitas")

# 2. Top 5
print(f"\n=== TOP 5 URLs ===")
for i, (url, count) in enumerate(reduced_urls[:5], 1):
    print(f"  {i}. {url} ({count:,})")

# 3. Errores por IP
print(f"\n=== ERRORES POR IP (status >= 400) ===")
mapped_errors = [map_errores_por_ip(log) for log in logs]
shuffled_errors = shuffle(mapped_errors)
reduced_errors = [reduce_sum(k, v) for k, v in shuffled_errors.items()]
reduced_errors.sort(key=lambda x: x[1], reverse=True)

for ip, count in reduced_errors[:10]:
    print(f"  {ip:20s} → {count:,} errores")

---

## Ejercicio 3: Simulación Pig Latin con Pandas (15 puntos)

Traduce las siguientes operaciones Pig Latin a su equivalente con Pandas:

```pig
-- Pig Latin
ventas = LOAD 'ventas.csv' USING CSVLoader AS (producto:chararray, cantidad:int, precio:float, fecha:chararray);
filtradas = FILTER ventas BY cantidad > 5;
agrupadas = GROUP filtradas BY producto;
resultado = FOREACH agrupadas GENERATE group AS producto, SUM(filtradas.cantidad) AS total_qty, AVG(filtradas.precio) AS precio_medio;
ordenado = ORDER resultado BY total_qty DESC;
top10 = LIMIT ordenado 10;
STORE top10 INTO 'resultado';
```

In [ ]:
import pandas as pd
import numpy as np

# LOAD: crear datos simulados
np.random.seed(42)
n = 5000
productos = [f"Producto_{chr(65+i)}" for i in range(15)]

ventas = pd.DataFrame({
    'producto': np.random.choice(productos, n),
    'cantidad': np.random.randint(1, 20, n),
    'precio': np.round(np.random.uniform(5.0, 100.0, n), 2),
    'fecha': pd.date_range('2025-01-01', periods=n, freq='h').strftime('%Y-%m-%d')
})

print(f"LOAD: {len(ventas)} registros cargados")
print(ventas.head())

# FILTER ventas BY cantidad > 5
filtradas = ventas[ventas['cantidad'] > 5]
print(f"\nFILTER: {len(filtradas)} registros tras filtrar cantidad > 5")

# GROUP BY producto + FOREACH GENERATE
resultado = filtradas.groupby('producto').agg(
    total_qty=('cantidad', 'sum'),
    precio_medio=('precio', 'mean')
).reset_index()
resultado['precio_medio'] = resultado['precio_medio'].round(2)
print(f"\nGROUP + FOREACH: {len(resultado)} grupos")

# ORDER BY total_qty DESC
ordenado = resultado.sort_values('total_qty', ascending=False)

# LIMIT 10
top10 = ordenado.head(10)

print(f"\nORDER + LIMIT 10:")
print(top10.to_string(index=False))

---

## Ejercicio 4: Simulación HiveQL con SQLite (15 puntos)

Traduce las siguientes consultas HiveQL a SQL y ejecútalas sobre SQLite:

```sql
-- HiveQL
-- 1. Ventas totales por producto y mes
SELECT producto,
       SUBSTR(fecha, 1, 7) AS mes,
       SUM(cantidad * precio) AS total_ventas
FROM ventas
GROUP BY producto, SUBSTR(fecha, 1, 7)
ORDER BY total_ventas DESC;

-- 2. Productos con ventas acumuladas > 10000
SELECT producto, SUM(cantidad * precio) AS acumulado
FROM ventas
GROUP BY producto
HAVING SUM(cantidad * precio) > 10000;
```

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')

# Cargar datos en SQLite
ventas.to_sql('ventas', conn, if_exists='replace', index=False)

# Consulta 1: Ventas totales por producto y mes
print("=== Consulta 1: Ventas por producto y mes ===")
query1 = """
SELECT producto,
       SUBSTR(fecha, 1, 7) AS mes,
       ROUND(SUM(cantidad * precio), 2) AS total_ventas
FROM ventas
GROUP BY producto, SUBSTR(fecha, 1, 7)
ORDER BY total_ventas DESC
LIMIT 15
"""
df1 = pd.read_sql(query1, conn)
print(df1.to_string(index=False))

# Consulta 2: Productos con acumulado > 10000
print("\n=== Consulta 2: Productos con acumulado > 10000 ===")
query2 = """
SELECT producto,
       ROUND(SUM(cantidad * precio), 2) AS acumulado
FROM ventas
GROUP BY producto
HAVING SUM(cantidad * precio) > 10000
ORDER BY acumulado DESC
"""
df2 = pd.read_sql(query2, conn)
print(df2.to_string(index=False))

conn.close()

---

## Ejercicio 5: Diseño de pipeline (10 puntos)

Diseña y codifica un pipeline completo que simule:

1. **Ingestión** (Sqoop): importar datos de una BBDD relacional
2. **Almacenamiento** (HDFS): dividir en bloques
3. **Procesamiento** (MapReduce): agregar datos
4. **Resultado** (Hive): consulta SQL sobre el resultado

In [ ]:
import math
import json

print("="*60)
print("PIPELINE COMPLETO: Sqoop → HDFS → MapReduce → Hive")
print("="*60)

# --- PASO 1: SQOOP (importar desde RDBMS) ---
print("\n📥 PASO 1: Importación (simula Sqoop)")

conn = sqlite3.connect(':memory:')
conn.execute('''
    CREATE TABLE transacciones (
        id INTEGER PRIMARY KEY,
        cliente TEXT,
        producto TEXT,
        importe REAL,
        fecha TEXT
    )
''')

import random
random.seed(42)
clientes = [f"Cliente_{i:03d}" for i in range(50)]
productos = ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",
             "Webcam", "SSD", "RAM", "Cable", "Hub USB"]

datos = [(i, random.choice(clientes), random.choice(productos),
          round(random.uniform(10, 500), 2),
          f"2026-01-{random.randint(1,31):02d}")
         for i in range(10000)]

conn.executemany('INSERT INTO transacciones VALUES (?,?,?,?,?)', datos)
conn.commit()

df_importado = pd.read_sql('SELECT * FROM transacciones', conn)
print(f"  Importados {len(df_importado):,} registros desde RDBMS")

# --- PASO 2: HDFS (split en bloques) ---
print("\n💾 PASO 2: Almacenamiento en bloques (simula HDFS)")

registros_por_bloque = 2000
num_bloques = math.ceil(len(df_importado) / registros_por_bloque)
bloques = [df_importado.iloc[i*registros_por_bloque:(i+1)*registros_por_bloque]
           for i in range(num_bloques)]

print(f"  Fichero dividido en {num_bloques} bloques")
for i, bloque in enumerate(bloques):
    print(f"  Bloque {i}: {len(bloque)} registros")

# --- PASO 3: MAPREDUCE ---
print("\n⚙️ PASO 3: Procesamiento (simula MapReduce)")

# Map: procesar cada bloque
all_mapped = []
for i, bloque in enumerate(bloques):
    mapped = [(row['producto'], row['importe']) for _, row in bloque.iterrows()]
    all_mapped.extend(mapped)
    print(f"  Mapper {i}: procesados {len(mapped)} registros")

# Shuffle
shuffled = defaultdict(list)
for producto, importe in all_mapped:
    shuffled[producto].append(importe)

# Reduce
resultados_mr = []
for producto, importes in shuffled.items():
    resultados_mr.append({
        'producto': producto,
        'total_ventas': round(sum(importes), 2),
        'num_transacciones': len(importes),
        'ticket_medio': round(sum(importes)/len(importes), 2)
    })

print(f"  Reduce: {len(resultados_mr)} grupos generados")

# --- PASO 4: HIVE (consulta SQL) ---
print("\n📊 PASO 4: Consulta (simula Hive)")

df_resultado = pd.DataFrame(resultados_mr)
df_resultado.to_sql('ventas_agregadas', conn, if_exists='replace', index=False)

query = """
SELECT producto, total_ventas, num_transacciones, ticket_medio
FROM ventas_agregadas
ORDER BY total_ventas DESC
"""

print(pd.read_sql(query, conn).to_string(index=False))

conn.close()
print("\n" + "="*60)
print("✅ Pipeline completado")
print("="*60)

---

## Resumen de puntuación

| Ejercicio | Tema | Puntos |
|-----------|------|--------|
| 1 | HDFS: bloques y replicación | 15 |
| 2 | MapReduce: análisis de logs | 20 |
| 3 | Pig Latin → Pandas | 15 |
| 4 | HiveQL → SQLite | 15 |
| 5 | Pipeline completo | 10 |
| **Total** | | **75** |

Nota: Los 25 puntos restantes corresponden a la prueba escrita teórica.

---

## Recursos

- [Hadoop Documentation](https://hadoop.apache.org/docs/current/)
- [Pig Tutorial](https://pig.apache.org/docs/latest/start.html)
- [Hive Language Manual](https://cwiki.apache.org/confluence/display/Hive/LanguageManual)